## This is a notebook for our churn prediction model pipeline

### Connect to Google BigQuery Database

### Look at features - EDA

### Feature engineering - Customer segmentation (Kmeans), Time series analysis

### Model selection

### Model evaulation


In [1]:
import gc
import time
import platform
import os
from dataclasses import dataclass
from typing import Callable, Any, List, Dict

import numpy as np
import pandas as pd

from google.cloud import bigquery
import db_dtypes


### Configure bigquery credentials

#### you may need to download gcloud - run 'brew install --cask google-cloud-sdk' and follow instructions to add to path

#### change the sql query to get different joined tables


In [2]:
PROJECT_ID = 'netflix-user-behavior'
# can add different queries based on what we need. 
# project id stays the same 
SQL = os.environ.get(
    "BQ_SQL",
    """
    SELECT
    m.*,
    w.*
FROM `netflix-user-behavior.kaggle_cleaned.movies_cleaned` AS m
JOIN `netflix-user-behavior.kaggle_cleaned.watch_history_cleaned` AS w
    ON m.movie_id = w.movie_id
LIMIT 1000;
    """.strip()
)
SQL_USER_BEHAVIOR = os.environ.get(
	"BQ_SQL_USER_BEHAVIOR",

	"""
	SELECT * 
	FROM `netflix-user-behavior.kaggle_cleaned.users_cleaned`
	"""
)
print("PROJECT_ID:", PROJECT_ID)
print("SQL preview:\n", SQL[:300], "..." if len(SQL) > 300 else "")
print("SQL_USER_BEHAVIOR preview:\n", SQL_USER_BEHAVIOR[:300], "..." if len(SQL_USER_BEHAVIOR) > 300 else "")

PROJECT_ID: netflix-user-behavior
SQL preview:
 SELECT
    m.*,
    w.*
FROM `netflix-user-behavior.kaggle_cleaned.movies_cleaned` AS m
JOIN `netflix-user-behavior.kaggle_cleaned.watch_history_cleaned` AS w
    ON m.movie_id = w.movie_id
LIMIT 1000; 
SQL_USER_BEHAVIOR preview:
 
	SELECT * 
	FROM `netflix-user-behavior.kaggle_cleaned.users_cleaned`
	 


In [3]:
# load in data from bigquery to pandas dataframe.
def load_bigquery_to_pandas(project_id: str, sql: str) -> pd.DataFrame:
	# authenticates using google credentials and connects to bigquery
	client = bigquery.Client(project=project_id)
    # sends request to bigquery and returns QueryJob object
	job = client.query(sql)
	df = job.result().to_dataframe()
	return df

#
pdf = load_bigquery_to_pandas(PROJECT_ID, SQL)
user = load_bigquery_to_pandas(PROJECT_ID, SQL_USER_BEHAVIOR)

c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\auth\_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\auth\_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting:

In [ ]:
user.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6735 entries, 0 to 6734
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype              
---  ------                   --------------  -----              
 0   user_id                  6735 non-null   object             
 1   email                    6735 non-null   object             
 2   first_name               6735 non-null   object             
 3   last_name                6735 non-null   object             
 4   age                      6735 non-null   float64            
 5   gender                   6735 non-null   object             
 6   country                  6735 non-null   object             
 7   state_province           6735 non-null   object             
 8   city                     6735 non-null   object             
 9   subscription_plan        6735 non-null   object             
 10  subscription_start_date  6735 non-null   dbdate             
 11  is_active                6735 

In [5]:
# pdf = watch_history + movies 
print(pdf.columns)
pdf.head()

Index(['movie_id', 'title', 'content_type', 'genre_primary', 'genre_secondary',
       'release_year', 'duration_minutes', 'rating', 'language',
       'country_of_origin', 'imdb_rating', 'is_netflix_original',
       'added_to_platform', 'content_warning', 'is_series', 'session_id',
       'user_id', 'movie_id_1', 'watch_date', 'device_type',
       'watch_duration_minutes', 'progress_percentage', 'action', 'quality',
       'location_country', 'is_download', 'has_rated'],
      dtype='object')


,movie_id,title,content_type,genre_primary,genre_secondary,release_year,duration_minutes,rating,language,country_of_origin,...,movie_id_1,watch_date,device_type,watch_duration_minutes,progress_percentage,action,quality,location_country,is_download,has_rated
0,movie_0140,City Empire,Documentary,Sport,None,2023,19.0,TV-14,English,France,...,movie_0140,2024-04-19,Desktop,51.2,31.2,started,SD,USA,False,0
1,movie_0140,City Empire,Documentary,Sport,None,2023,19.0,TV-14,English,France,...,movie_0140,2024-05-12,Desktop,110.0,94.1,started,4K,Canada,False,1
2,movie_0140,City Empire,Documentary,Sport,None,2023,19.0,TV-14,English,France,...,movie_0140,2025-03-18,Desktop,41.6,73.7,stopped,HD,USA,False,0
3,movie_0140,City Empire,Documentary,Sport,None,2023,19.0,TV-14,English,France,...,movie_0140,2025-03-22,Desktop,88.4,94.8,started,4K,USA,False,0
4,movie_0140,City Empire,Documentary,Sport,None,2023,19.0,TV-14,English,France,...,movie_0140,2025-05-03,Desktop,11.6,99.9,started,SD,USA,False,1


In [6]:
# Merge user and pdf based on user id
merged_df = pd.merge(user, pdf, on='user_id', how='left')

#  (we can figure out users who never used the service) 
print(f"Total users: {len(user)}")
print(f"The number of rows after join two tables: {len(merged_df)}")
merged_df.head()

Total users: 6735
The number of rows after join two tables: 6808


,user_id,email,first_name,last_name,age,gender,country,state_province,city,subscription_plan,...,movie_id_1,watch_date,device_type,watch_duration_minutes,progress_percentage,action,quality,location_country,is_download,has_rated
0,user_00784,alexander25@example.org,Stacey,Cortez,25.0,Female,Canada,Alberta,Guzmanburgh,Basic,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
1,user_00988,sarahrollins@example.com,Evelyn,Hayes,33.0,Female,Canada,Alberta,East Elizabeth,Standard,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
2,user_01621,jeffreyfinley@example.org,Patrick,Hayes,55.0,Male,Canada,Alberta,West Christian,Basic,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
3,user_01821,grimeshenry@example.net,David,Trevino,31.0,Female,Canada,Alberta,South Angela,Premium,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>
4,user_01956,michaelwood@example.org,William,Rush,39.0,Male,Canada,Alberta,New April,Standard,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>


Logistic Regression

In [41]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [33]:
# Created user-level features since watch-history creates duplicates
watch_features = merged_df.groupby("user_id").agg(
total_watch_time=("watch_duration_minutes", "sum"),
avg_progress=("progress_percentage", "mean"),
total_movies=("movie_id_1", "count"),
downloads=("is_download", "sum"),
ratings=("has_rated", "sum")    
).reset_index()

In [34]:
user_features = merged_df[[
    "user_id",
    "age",
    "gender",
    "country",
    "subscription_plan",
    "is_active"
]].drop_duplicates()
    
dataset = user_features.merge(watch_features, on="user_id", how="left")

In [35]:
dataset = pd.get_dummies(dataset, columns=["gender", "subscription_plan", "country"], drop_first=True)

In [42]:
# Target and features defined
dataset["churn"] = (~dataset["is_active"]).astype(int)

X = dataset.drop(columns=["user_id", "is_active", "churn"])
y = dataset["churn"]

X = pd.get_dummies(X, drop_first=True)

In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [45]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [48]:
# Train baseline logistic regression
model = HistGradientBoostingClassifier(random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

HistGradientBoostingClassifier(class_weight='balanced', random_state=42)

In [49]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Confusion Matrix:
 [[663 488]
 [111  85]]

Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.58      0.69      1151
           1       0.15      0.43      0.22       196

    accuracy                           0.56      1347
   macro avg       0.50      0.50      0.45      1347
weighted avg       0.75      0.56      0.62      1347

ROC-AUC: 0.5147276547456516


Rolling 7 day feature

Logistic Regression with selected features

If logistic regression does not perform well, we can try Random Forest!

### pull basic user information from the user table, how many unique users do we have and what is the time period we are looking at
